In [2]:
!pip install pulp
import pulp
prob = pulp.LpProblem("President_Goals", pulp.LpMinimize)

G = pulp.LpVariable("G", lowBound=0, cat="Continuous")       # Gas tax (cents per gallon)
T1 = pulp.LpVariable("T1", lowBound=0, cat="Continuous")     # Tax rate on first $30,000
T2 = pulp.LpVariable("T2", lowBound=0, cat="Continuous")     # Tax rate on income over $30,000
C = pulp.LpVariable("C", lowBound=0, cat="Continuous")       # Spending cut (in billions)

d1_neg = pulp.LpVariable("d1_neg", lowBound=0, cat="Continuous")  # Unmet budget
d2_pos = pulp.LpVariable("d2_pos", lowBound=0, cat="Continuous")  # Excess spending cut
d3_pos = pulp.LpVariable("d3_pos", lowBound=0, cat="Continuous")  # Excess upper-class tax
d4_pos = pulp.LpVariable("d4_pos", lowBound=0, cat="Continuous")  # Excess lower-class tax

# T2 >= T1
prob += T2 >= T1, "Tax_Rate_Constraint"

# Goal 1:
prob += G + 0.5*G + 20*T1 + 5*T1 + 15*T2 + C - d1_neg >= 1000, "Budget_Balance"

# Goal 2:
prob += C - d2_pos <= 150, "Spending_Cut"

# Goal 3:
prob += 0.5*G + 5*T1 + 15*T2 - d3_pos <= 550, "Upper_Tax"

# Goal 4:
prob += G + 20*T1 - d4_pos <= 350, "Lower_Tax"


# Goal 1: Minimize d1_neg
prob.setObjective(d1_neg)  # Objective for Goal 1
prob.solve()
print("Goal 1: Balance the Budget")
print(f"Status: {pulp.LpStatus[prob.status]}")
for var in prob.variables():
    print(f"{var.name} = {var.varValue}")
print(f"Objective (d1_neg) = {pulp.value(prob.objective)}\n")

# Goal 2: Minimize d2_pos (Excess Spending Cut)
prob += d1_neg == 0  # Fixing d1_neg to ensure Goal 1 remains satisfied
prob.setObjective(d2_pos)  # Objective for Goal 2
prob.solve()
print("Goal 2: Cut Spending ")
print(f"Status: {pulp.LpStatus[prob.status]}")
for var in prob.variables():
    print(f"{var.name} = {var.varValue}")
print(f"Objective (d2_pos) = {pulp.value(prob.objective)}\n")

# Goal 3: Minimize d3_pos (Excess Upper-Class Tax)
prob += d2_pos == 0  # Fixing d2_pos to ensure Goal 2 remains satisfied
prob.setObjective(d3_pos)  # Objective for Goal 3
prob.solve()
print("Goal 3: Upper-Class Tax")
print(f"Status: {pulp.LpStatus[prob.status]}")
for var in prob.variables():
    print(f"{var.name} = {var.varValue}")
print(f"Objective (d3_pos) = {pulp.value(prob.objective)}\n")

# Goal 4: Minimize d4_pos (Excess Lower-Class Tax)
prob += d3_pos == 0  # Fix d3_pos to ensure Goal 3 remains satisfied
prob.setObjective(d4_pos)  # Objective for Goal 4
prob.solve()
print("Goal 4: Lower-Class Tax ")
print(f"Status: {pulp.LpStatus[prob.status]}")
for var in prob.variables():
    print(f"{var.name} = {var.varValue}")
print(f"Objective (d4_pos) = {pulp.value(prob.objective)}\n")

#Results
print("Final Solution")
for var in prob.variables():
    print(f"{var.name} = {var.varValue}")



Goal 1: Balance the Budget
Status: Optimal
C = 0.0
G = 0.0
T1 = 25.0
T2 = 25.0
d1_neg = 0.0
d2_pos = 0.0
d3_pos = 0.0
d4_pos = 150.0
Objective (d1_neg) = 0.0

Goal 2: Cut Spending 
Status: Optimal
C = 150.0
G = 0.0
T1 = 21.25
T2 = 21.25
d1_neg = 0.0
d2_pos = 0.0
d3_pos = 0.0
d4_pos = 75.0
Objective (d2_pos) = 0.0

Goal 3: Upper-Class Tax
Status: Optimal
C = 150.0
G = 0.0
T1 = 27.5
T2 = 27.5
d1_neg = 0.0
d2_pos = 0.0
d3_pos = 0.0
d4_pos = 200.0
Objective (d3_pos) = 0.0

Goal 4: Lower-Class Tax 
Status: Optimal
C = 150.0
G = 300.0
T1 = 0.0
T2 = 26.666667
d1_neg = 0.0
d2_pos = 0.0
d3_pos = 0.0
d4_pos = 0.0
Objective (d4_pos) = 0.0

Final Solution
C = 150.0
G = 300.0
T1 = 0.0
T2 = 26.666667
d1_neg = 0.0
d2_pos = 0.0
d3_pos = 0.0
d4_pos = 0.0


b - **Conduct a sensitivity analysis to determine how sensitive the revenue is to changes in G, T1, and T2. Which variable has the most significant impact on total revenue?**

In [1]:
# Define the base values and total revenue calculation function
C_base = 150.0  # Spending cut (in billions)
G_base = 300.0  # Gas tax (cents per gallon)
T1_base = 0.0   # Tax rate on the first $30,000
T2_base = 26.666667  # Tax rate on income over $30,000

def calculate_total_revenue(G, T1, T2):
    R_gas = G + 0.5 * G  # Gas tax revenue
    R_low_income = 20 * T1  # Revenue from tax on the first $30,000
    R_high_income = 15 * T2  # Revenue from tax on income over $30,000
    return R_gas + R_low_income + R_high_income

# Base total revenue
R_total_base = calculate_total_revenue(G_base, T1_base, T2_base)

# Function to compute sensitivity for each variable
def sensitivity_analysis(base_value, percent_change, variable_name):
    # Adjust the variable by the specified percentage change
    new_value = base_value * (1 + percent_change)
    if variable_name == "G":
        new_revenue = calculate_total_revenue(new_value, T1_base, T2_base)
    elif variable_name == "T1":
        new_revenue = calculate_total_revenue(G_base, new_value, T2_base)
    elif variable_name == "T2":
        new_revenue = calculate_total_revenue(G_base, T1_base, new_value)
    else:
        raise ValueError("Invalid variable name.")

    # Compute the change in revenue
    delta = new_revenue - R_total_base
    return delta

# Percentage change for analysis 1%
percent_change = 0.01

# Perform sensitivity analysis for G, T1, and T2
delta_G = sensitivity_analysis(G_base, percent_change, "G")
delta_T1 = sensitivity_analysis(T1_base, percent_change, "T1")
delta_T2 = sensitivity_analysis(T2_base, percent_change, "T2")

# Compile results
results = {
    "G (Gas Tax)": delta_G,
    "T1 (Tax Rate on $30k)": delta_T1,
    "T2 (Tax Rate above $30k)": delta_T2
}

# Find the variable with the most significant impact
most_significant = max(results, key=results.get)

# Print results
print("Sensitivity Analysis Results:")
for variable, delta in results.items():
    print(f"{variable}: Change in Revenue = {delta:.2f} billion")

print(f"\nVariable with the most significant impact: {most_significant}")



Sensitivity Analysis Results:
G (Gas Tax): Change in Revenue = 4.50 billion
T1 (Tax Rate on $30k): Change in Revenue = 0.00 billion
T2 (Tax Rate above $30k): Change in Revenue = 4.00 billion

Variable with the most significant impact: G (Gas Tax)
